# Fantasy Football Weekly Projections (Half-PPR)

Generates weekly half-PPR fantasy projections for QB, RB, WR, TE using per-position XGBoost models.
Also outputs per-stat prop columns (pass yds, rush yds, receptions, rec yds) for use in sports betting prop research.

Run via **papermill** from the project root:
```bash
papermill fantasy/predict_fantasy.ipynb /tmp/out.ipynb                                      # auto-detect week
papermill fantasy/predict_fantasy.ipynb /tmp/out.ipynb -p TARGET_SEASON 2025 -p TARGET_WEEK 14
```
Output saved to `fantasy/fantasy_projections/projections_{season}_week{week:02d}.csv`.

**Steps:**
1. Load main position models + 8 per-stat prop models
2. Auto-detect or use the specified target week
3. Pull schedule context, player rolling history, injuries, and depth charts
4. Score all active players with per-position and per-stat models
5. Save CSV to `fantasy/fantasy_projections/`
6. **Step 7** — run the projection analysis cell for distribution summary, prop stat leaders, and position scorecards
7. **Model Performance Summary** (last cell) — 2025 weeks 10–17 MAE/bias/correlation benchmarks by position


## Parameters

In [10]:
TARGET_SEASON = 2025
TARGET_WEEK   = None  # None = auto-detect next unplayed week
POS_FILTER    = None  # None | "QB" | "RB" | "WR" | "TE"

## Setup — Imports & Config

In [ ]:
# Polars >= 1.x is strict about UTF-8 in parquet files; nflverse data
# occasionally contains non-UTF-8 strings. Fall back to pyarrow on failure.
import polars as _pl
_pl_read_parquet_orig = _pl.read_parquet
def _pl_read_parquet_lenient(source, *args, **kwargs):
    try:
        return _pl_read_parquet_orig(source, *args, **kwargs)
    except Exception:
        kwargs.setdefault('use_pyarrow', True)
        return _pl_read_parquet_orig(source, *args, **kwargs)
_pl.read_parquet = _pl_read_parquet_lenient

import json
import warnings
import joblib
import numpy as np
import pandas as pd
import nflreadpy as nfl
from pathlib import Path
import os

warnings.filterwarnings("ignore")

# Works whether kernel starts from project root or from fantasy/ directly
_cwd         = Path.cwd()
_DIR         = _cwd if _cwd.name == "fantasy" else _cwd / "fantasy"
FEATURES_CSV = _DIR / "features_dataset.csv"
MODEL_DIR    = _DIR / "models"
POSITIONS    = ["QB", "RB", "WR", "TE"]

INJURY_MAP   = {"Out": 0.0, "Doubtful": 0.1, "Questionable": 0.5, "Probable": 0.75}
PRACTICE_MAP = {"Did Not Participate In Practice": 0.0, "Limited Participation in Practice": 0.5, "Full Participation in Practice": 1.0}

TURF_SURFACES = {"astroturf", "fieldturf", "turf", "matrixturf", "sportturf", "astroplay", "a_turf"}

## Step 1 — Load Models

Loads the four main position models (`models/{pos}_model.pkl`) plus eight per-stat prop models:

| Model file | Predicts | 2025 MAE |
|---|---|---|
| `qb_pass_yards_model.pkl` | QB passing yards | 72.5 yds |
| `qb_rush_yards_model.pkl` | QB rushing yards | 12.3 yds |
| `rb_rush_yards_model.pkl` | RB rushing yards | 21.1 yds |
| `rb_rec_yards_model.pkl` | RB receiving yards | 10.3 yds |
| `wr_receptions_model.pkl` | WR receptions | 1.4 rec |
| `wr_rec_yards_model.pkl` | WR receiving yards | 21.2 yds |
| `te_receptions_model.pkl` | TE receptions | 1.3 rec |
| `te_rec_yards_model.pkl` | TE receiving yards | 15.5 yds |

QB models use the reduced QB feature set (receiving cols excluded). All others use the full feature set.

In [19]:
# â”€â”€ Load models â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
models = {}
for pos in POSITIONS:
    saved = joblib.load(MODEL_DIR / f"{pos.lower()}_model.pkl")
    models[pos] = saved
print("Models loaded:", {p: len(m["feature_cols"]) for p, m in models.items()})

# ── QB per-stat models ────────────────────────────────────────────
QB_STAT_NAMES  = ["pass_yards", "rush_yards"]
QB_STAT_MODELS = {}
for stat in QB_STAT_NAMES:
    pkl_path = MODEL_DIR / f"qb_{stat}_model.pkl"
    if pkl_path.exists():
        QB_STAT_MODELS[stat] = joblib.load(pkl_path)
print("QB stat models loaded:", list(QB_STAT_MODELS.keys()))

# ── RB per-stat models ────────────────────────────────────────────
RB_STAT_NAMES  = ["rush_yards", "rec_yards"]
RB_STAT_MODELS = {}
for stat in RB_STAT_NAMES:
    pkl_path = MODEL_DIR / f"rb_{stat}_model.pkl"
    if pkl_path.exists():
        RB_STAT_MODELS[stat] = joblib.load(pkl_path)
print("RB stat models loaded:", list(RB_STAT_MODELS.keys()))

# ── WR per-stat models ────────────────────────────────────────────
WR_STAT_NAMES  = ["receptions", "rec_yards"]
WR_STAT_MODELS = {}
for stat in WR_STAT_NAMES:
    pkl_path = MODEL_DIR / f"wr_{stat}_model.pkl"
    if pkl_path.exists():
        WR_STAT_MODELS[stat] = joblib.load(pkl_path)
print("WR stat models loaded:", list(WR_STAT_MODELS.keys()))

# ── TE per-stat models ────────────────────────────────────────────
TE_STAT_NAMES  = ["receptions", "rec_yards"]
TE_STAT_MODELS = {}
for stat in TE_STAT_NAMES:
    pkl_path = MODEL_DIR / f"te_{stat}_model.pkl"
    if pkl_path.exists():
        TE_STAT_MODELS[stat] = joblib.load(pkl_path)
print("TE stat models loaded:", list(TE_STAT_MODELS.keys()))

# â”€â”€ Determine target season / week â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

Models loaded: {'QB': 61, 'RB': 84, 'WR': 84, 'TE': 84}
QB stat models loaded: ['pass_yards', 'rush_yards']
RB stat models loaded: ['rush_yards', 'rec_yards']
WR stat models loaded: ['receptions', 'rec_yards']
TE stat models loaded: ['receptions', 'rec_yards']


## Step 2 — Detect Target Week

In [20]:
def detect_week(season):
    raw   = nfl.load_schedules([season])
    sched = raw.to_pandas() if hasattr(raw, "to_pandas") else pd.DataFrame(raw)
    reg   = sched[(sched["season"] == season) & (sched["game_type"] == "REG")]
    future = reg[reg["result"].isna()]
    return int(future["week"].min()) if not future.empty else None


if TARGET_WEEK is None:
    TARGET_WEEK = detect_week(TARGET_SEASON)
if TARGET_WEEK is None:
    raise ValueError(f"Season {TARGET_SEASON} complete â€” no upcoming games.")

print(f"\nProjecting Season {TARGET_SEASON}  Week {TARGET_WEEK}"
      + (f"  [{POS_FILTER}]" if POS_FILTER else "  [all positions]"))


Projecting Season 2025  Week 10  [all positions]


## Step 3 — Upcoming Schedule

In [21]:
# â”€â”€ Load upcoming schedule â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
raw_sched = nfl.load_schedules([TARGET_SEASON])
schedule  = raw_sched.to_pandas() if hasattr(raw_sched, "to_pandas") else pd.DataFrame(raw_sched)
schedule["season"] = schedule["season"].astype(int)
schedule["week"]   = schedule["week"].astype(int)

upcoming = schedule[
    (schedule["season"] == TARGET_SEASON) &
    (schedule["week"]   == TARGET_WEEK) &
    (schedule["game_type"] == "REG")
].copy()

if upcoming.empty:
    raise ValueError(f"No REG games found for week {TARGET_WEEK}.")

print(f"{len(upcoming)} games found.")

# Build one row per team per game
home = upcoming[["game_id", "home_team", "away_team", "spread_line", "total_line",
                  "roof", "surface", "temp", "wind", "home_rest", "away_rest", "gameday"]].copy()
home.rename(columns={"home_team": "team", "away_team": "opponent_team", "home_rest": "rest"}, inplace=True)
home.drop(columns=["away_rest"], inplace=True, errors="ignore")
home["is_home"] = 1

away = upcoming[["game_id", "away_team", "home_team", "spread_line", "total_line",
                  "roof", "surface", "temp", "wind", "home_rest", "away_rest", "gameday"]].copy()
away.rename(columns={"away_team": "team", "home_team": "opponent_team", "away_rest": "rest"}, inplace=True)
away.drop(columns=["home_rest"], inplace=True, errors="ignore")
away["is_home"] = 0

team_ctx = pd.concat([home, away], ignore_index=True)

team_ctx["temp"]  = team_ctx["temp"].fillna(72)
team_ctx["wind"]  = team_ctx["wind"].fillna(0)
team_ctx["is_dome"]  = team_ctx["roof"].isin(["dome", "closed"]).astype(int)
team_ctx["is_turf"]  = team_ctx["surface"].isin(TURF_SURFACES).astype(int)
team_ctx["effective_wind"] = np.where(team_ctx["is_dome"], 0,  team_ctx["wind"])
team_ctx["effective_temp"] = np.where(team_ctx["is_dome"], 72, team_ctx["temp"])
team_ctx["days_rest"] = team_ctx["rest"].fillna(7)

# implied_team_total: home = (total - spread) / 2, away = (total + spread) / 2
# spread_line is home-perspective (negative = home favored), same as features_dataset
team_ctx["implied_team_total"] = np.where(
    team_ctx["is_home"] == 1,
    (team_ctx["total_line"] - team_ctx["spread_line"]) / 2,
    (team_ctx["total_line"] + team_ctx["spread_line"]) / 2,
)

ctx_cols = ["team", "opponent_team", "is_home", "spread_line", "total_line",
            "implied_team_total", "days_rest", "is_dome", "is_turf",
            "effective_wind", "effective_temp", "gameday", "game_id"]

14 games found.


## Step 4 — Player History & Live Defensive Metrics

- **Player history**: loads `features_dataset.csv`, takes each player's most recent row as their current rolling form. Drops players last seen 2+ seasons ago (retired/cut).
- **Defensive metrics (live)**: loads PBP for `TARGET_SEASON` via `nfl.load_pbp()`, filters to completed games (`week < TARGET_WEEK`), aggregates per-game defensive stats (EPA allowed, yards allowed, pass rate faced, red zone rate) by `defteam`, takes each team’s last 4 games and computes rolling means. Falls back to `features_dataset.csv` lookup if PBP is unavailable.
- **Game context**: merges player rows with upcoming schedule context from `team_ctx` (spread, total, weather, home/away) and live `opp_def` defensive metrics by `opponent_team`.

In [22]:
# ── Load player history — each player's latest rolling form ────────────────
hist = pd.read_csv(FEATURES_CSV)
hist["season"] = hist["season"].astype(int)
hist["week"]   = hist["week"].astype(int)

# Most recent completed row per player = current rolling form
latest = (
    hist.sort_values(["player_id", "season", "week"])
    .groupby("player_id").last().reset_index()
)

# Drop players who haven't appeared in the last two seasons (retired/cut)
latest = latest[latest["season"] >= TARGET_SEASON - 1]

# Live defensive metrics from PBP — replaces stale features_dataset lookup.
# Computes rolling 4-game average of each team's defensive performance using
# only completed games this season (week < TARGET_WEEK).
DEF_COLS = ["def_epa_allowed_roll4", "def_yards_allowed_roll4",
            "def_pass_rate_faced_roll4", "def_red_zone_allowed_roll4"]
_DEF_RAW  = ["epa_allowed_per_play", "yards_allowed_per_play",
             "pass_rate_faced",       "rz_allowed_rate"]

try:
    _pbp_raw = nfl.load_pbp([TARGET_SEASON])
    _pbp = _pbp_raw.to_pandas() if hasattr(_pbp_raw, "to_pandas") else pd.DataFrame(_pbp_raw)
    _pbp = _pbp[_pbp["play_type"].isin(["run", "pass"]) & _pbp["posteam"].notna()].copy()
    _pbp["season"] = _pbp["season"].astype(int)
    _pbp["week"]   = _pbp["week"].astype(int)
    _pbp = _pbp[_pbp["week"] < TARGET_WEEK]  # completed games only

    _pbp["is_pass"] = (_pbp["play_type"] == "pass").astype(int)
    _pbp["is_rz"]   = (_pbp["yardline_100"] <= 20).astype(int)
    _pbp["is_rz_td"]= ((_pbp["yardline_100"] <= 20) & (_pbp["touchdown"] == 1)).astype(int)

    _def_game = _pbp.groupby(["week", "defteam"]).agg(
        epa_sum    =("epa",          "sum"),
        yards_sum  =("yards_gained", "sum"),
        play_count =("play_id",      "count"),
        pass_count =("is_pass",      "sum"),
        rz_plays   =("is_rz",        "sum"),
        rz_tds     =("is_rz_td",     "sum"),
    ).reset_index().rename(columns={"defteam": "team"})

    _def_game["epa_allowed_per_play"]   = _def_game["epa_sum"]   / _def_game["play_count"]
    _def_game["yards_allowed_per_play"] = _def_game["yards_sum"]  / _def_game["play_count"]
    _def_game["pass_rate_faced"]        = _def_game["pass_count"] / _def_game["play_count"]
    _def_game["rz_allowed_rate"]        = (
        _def_game["rz_tds"] / _def_game["rz_plays"].replace(0, float("nan"))
    )

    # Last 4 completed games per team → mean (rolling window consistent with training)
    _def_last4 = _def_game.sort_values(["team", "week"]).groupby("team").tail(4)
    opp_def = (
        _def_last4.groupby("team")[_DEF_RAW].mean().reset_index()
        .rename(columns={"team":                  "opponent_team",
                         "epa_allowed_per_play":  "def_epa_allowed_roll4",
                         "yards_allowed_per_play":"def_yards_allowed_roll4",
                         "pass_rate_faced":       "def_pass_rate_faced_roll4",
                         "rz_allowed_rate":       "def_red_zone_allowed_roll4"})
    )
    # Fill red zone rate NaN (games with zero red zone plays) with league average
    if 'def_red_zone_allowed_roll4' in opp_def.columns:
        opp_def['def_red_zone_allowed_roll4'] = opp_def['def_red_zone_allowed_roll4'].fillna(0.58)
    print(f"Live defensive metrics: {len(opp_def)} teams, weeks 1–{TARGET_WEEK - 1}")
except Exception as e:
    print(f"PBP unavailable ({e}) — falling back to features_dataset defensive lookup")
    opp_def = (
        hist[["opponent_team", "season", "week"] + DEF_COLS]
        .sort_values(["opponent_team", "season", "week"])
        .groupby("opponent_team").last().reset_index()
        [["opponent_team"] + DEF_COLS]
    )

# ── Join players with upcoming game context ────────────────────────────────────
active_teams = team_ctx["team"].tolist()
players = latest[latest["team"].isin(active_teams)].copy()
if POS_FILTER:
    players = players[players["position"] == POS_FILTER]

drop_cols = [c for c in ctx_cols + DEF_COLS if c != "team"]
players.drop(columns=drop_cols, errors="ignore", inplace=True)

players = players.merge(team_ctx[ctx_cols], on="team", how="inner")
players = players.merge(opp_def, on="opponent_team", how="left")
print(f"Players matched to upcoming games: {len(players)}")

# Live coach win% and opp season win% — replace stale features_dataset values
_COACH_COLS = ["coach_win_pct", "opp_coach_win_pct", "is_new_coach", "opp_is_new_coach"]
_SOS_COL    = "opp_season_win_pct"
_TEAM_NORM  = {"STL": "LA", "LAR": "LA", "OAK": "LV", "LVR": "LV",
               "SD": "LAC", "SDG": "LAC", "NWE": "NE", "KAN": "KC",
               "GNB": "GB", "NOR": "NO", "TAM": "TB", "SFO": "SF"}

try:
    # Load full schedule history for coach career records
    _sraw = nfl.load_schedules(list(range(1999, TARGET_SEASON + 1)))
    _sall = _sraw.to_pandas() if hasattr(_sraw, "to_pandas") else pd.DataFrame(_sraw)
    _sall["season"] = _sall["season"].astype(int)
    _sall["week"]   = _sall["week"].astype(int)

    _reg = _sall[(_sall["game_type"] == "REG") & _sall["result"].notna()].copy()
    for _col in ["home_team", "away_team"]:
        _reg[_col] = _reg[_col].replace(_TEAM_NORM)

    # Long format: one row per team per game
    _hc = _reg[["season", "week", "home_team", "home_score", "away_score", "home_coach"]].rename(
        columns={"home_team": "team", "home_score": "ts", "away_score": "os", "home_coach": "coach"})
    _ac = _reg[["season", "week", "away_team", "away_score", "home_score", "away_coach"]].rename(
        columns={"away_team": "team", "away_score": "ts", "home_score": "os", "away_coach": "coach"})
    _g = pd.concat([_hc, _ac], ignore_index=True)
    _g["win"] = (_g["ts"] > _g["os"]).astype(int)
    _g = _g.sort_values(["coach", "season", "week"]).reset_index(drop=True)
    _g["_cum_wins"]  = _g.groupby("coach")["win"].transform(lambda x: x.cumsum().shift(fill_value=0))
    _g["_cum_games"] = _g.groupby("coach").cumcount()
    # NaN for coaches with < 10 career games (flagged as new below)
    _g["coach_win_pct"] = (_g["_cum_wins"] / _g["_cum_games"]).where(_g["_cum_games"] >= 10)
    # Median from completed games only — excludes current-season inference rows
    _completed_mask = (_g["season"] < TARGET_SEASON) | (
        (_g["season"] == TARGET_SEASON) & (_g["week"] < TARGET_WEEK)
    )
    _coach_median = _g.loc[_completed_mask, "coach_win_pct"].median()

    # Coaches on the field for TARGET_WEEK
    _tw = _sall[(_sall["season"] == TARGET_SEASON) & (_sall["week"] == TARGET_WEEK)
                & (_sall["game_type"] == "REG")]
    _week_coaches = pd.concat([
        _tw[["home_team", "home_coach"]].rename(columns={"home_team": "team", "home_coach": "coach"}),
        _tw[["away_team", "away_coach"]].rename(columns={"away_team": "team", "away_coach": "coach"}),
    ]).dropna(subset=["coach"])

    # Each coach's career win% as of just before TARGET_WEEK
    _g_prev = _g[(_g["season"] < TARGET_SEASON) |
                 ((_g["season"] == TARGET_SEASON) & (_g["week"] < TARGET_WEEK))]
    _coach_wp = (
        _g_prev.sort_values(["coach", "season", "week"])
        .groupby("coach")["coach_win_pct"].last().reset_index()
    )
    _team_wp = _week_coaches.merge(_coach_wp, on="coach", how="left")
    _team_wp["is_new_coach"]  = _team_wp["coach_win_pct"].isna().astype(int)
    _team_wp["coach_win_pct"] = _team_wp["coach_win_pct"].fillna(_coach_median)

    players.drop(columns=_COACH_COLS, errors="ignore", inplace=True)
    players = players.merge(_team_wp[["team", "coach_win_pct", "is_new_coach"]], on="team", how="left")
    players = players.merge(
        _team_wp[["team", "coach_win_pct", "is_new_coach"]].rename(
            columns={"team": "opponent_team",
                     "coach_win_pct": "opp_coach_win_pct",
                     "is_new_coach":  "opp_is_new_coach"}),
        on="opponent_team", how="left"
    )
    for _c, _fill in [("coach_win_pct", _coach_median), ("opp_coach_win_pct", _coach_median),
                      ("is_new_coach", 0), ("opp_is_new_coach", 0)]:
        players[_c] = players[_c].fillna(_fill)
    players["is_new_coach"]     = players["is_new_coach"].astype(int)
    players["opp_is_new_coach"] = players["opp_is_new_coach"].astype(int)
    print(f"Live coach win%: {len(_team_wp)} teams")

    # Opponent current-season win% going into TARGET_WEEK
    _cs = _sall[(_sall["season"] == TARGET_SEASON) & (_sall["game_type"] == "REG")
                & (_sall["week"] < TARGET_WEEK) & _sall["home_score"].notna()].copy()
    _hw = pd.DataFrame({"week": _cs["week"], "team": _cs["home_team"],
                         "win": (_cs["home_score"] > _cs["away_score"]).astype(int)})
    _aw = pd.DataFrame({"week": _cs["week"], "team": _cs["away_team"],
                         "win": (_cs["away_score"] > _cs["home_score"]).astype(int)})
    _rec = pd.concat([_hw, _aw]).sort_values(["team", "week"]).reset_index(drop=True)
    _rec["cum_wins"]  = _rec.groupby("team")["win"].cumsum()
    _rec["cum_games"] = _rec.groupby("team").cumcount() + 1
    # Win% *going into* each week: exclude current game's result
    _rec["win_pct"] = ((_rec["cum_wins"] - _rec["win"])
                       / (_rec["cum_games"] - 1).replace(0, float("nan"))).fillna(0.5)
    _opp_wp_latest = (
        _rec.sort_values(["team", "week"]).groupby("team")["win_pct"].last()
        .reset_index().rename(columns={"team": "opponent_team", "win_pct": "opp_season_win_pct"})
    )
    players.drop(columns=[_SOS_COL], errors="ignore", inplace=True)
    players = players.merge(_opp_wp_latest, on="opponent_team", how="left")
    players[_SOS_COL] = players[_SOS_COL].fillna(0.5)
    print(f"Live opp_season_win_pct: {len(_opp_wp_latest)} teams")

except Exception as e:
    print(f"Schedule unavailable ({e}) — keeping features_dataset coach/SOS values")

Players matched to upcoming games: 590


## Step 5 - Injury & Depth Chart Refresh (shared adapter)

The FULL depth/availability contract is refreshed here, not just `depth_chart_position`:
`depth_chart_position`, `starter_{qb,rb,wr,te}_availability`, the eight
`opp_*1_availability` columns and `starter_{tackle,guard,center}_availability` - the same
sixteen columns, built by the same `depth_features` code, as at training time.

- **Depth charts**: `depth_features.build_live_depth_contract` normalises whichever
  nflverse schema is live, selects the newest snapshot **strictly before** this slate's
  first kickoff, and refuses a slate it cannot populate.
- **Fallback**: if the live fetch fails, `live_depth_snapshot` may use an explicitly
  RETAINED snapshot that carries provenance and is freshness-checked against the slate.
  There is no "last known values" path and no all-default path.
- **Injuries**: `nfl.load_injuries()` for `TARGET_SEASON`, filtered to `TARGET_WEEK`.
  Players with `injury_status_score == 0` (officially Out) are dropped before projecting.


In [ ]:
# -- Refresh injuries for this week -------------------------------------------
import sys
if "." not in sys.path:
    sys.path.insert(0, ".")
if str(_DIR) not in sys.path:
    sys.path.insert(0, str(_DIR))
import depth_features as DF
import live_depth_snapshot as LDS

raw_inj = nfl.load_injuries(seasons=[TARGET_SEASON])
injuries_all = raw_inj.to_pandas() if hasattr(raw_inj, "to_pandas") else pd.DataFrame(raw_inj)
inj_wk = injuries_all[injuries_all["week"] == TARGET_WEEK].copy()
if inj_wk.empty:
    raise RuntimeError(f"no injury report rows for {TARGET_SEASON} week {TARGET_WEEK} - "
                       "refusing to project on defaulted availability")
inj_wk["injury_status_score"] = inj_wk["report_status"].map(INJURY_MAP).fillna(1.0)
inj_wk["practice_status_score"] = inj_wk["practice_status"].map(PRACTICE_MAP).fillna(1.0)
inj_player = inj_wk[["gsis_id", "injury_status_score", "practice_status_score"]].rename(
    columns={"gsis_id": "player_id"}).drop_duplicates(subset=["player_id"], keep="first")
players.drop(columns=["injury_status_score", "practice_status_score"],
             errors="ignore", inplace=True)
players = players.merge(inj_player, on="player_id", how="left")
players["injury_status_score"] = players["injury_status_score"].fillna(1.0)
players["practice_status_score"] = players["practice_status_score"].fillna(1.0)
print(f"Injuries updated: {len(inj_wk)} report rows for week {TARGET_WEEK}.")

# -- Refresh the FULL depth / availability contract ----------------------------
# The slate cutoff is this week's FIRST kickoff; every snapshot used is strictly before
# it, so a retroactive run cannot see a post-promotion depth chart.
_slate_cutoff = pd.to_datetime(players["gameday"].min(), utc=True)

depth_raw, depth_prov = LDS.fetch_or_retained(
    lambda: nfl.load_depth_charts(seasons=[TARGET_SEASON]),
    season=TARGET_SEASON, cutoff_utc=_slate_cutoff)
print(f"Depth source: {depth_prov['provenance']} | rows={depth_prov['rows']} | "
      f"newest snapshot {depth_prov['max_snapshot_dt']}")

live = DF.build_live_depth_contract(
    depth_raw=depth_raw, injuries=injuries_all,
    season=TARGET_SEASON, week=TARGET_WEEK,
    cutoff_utc=_slate_cutoff, teams=players["team"].unique())
print(json.dumps({k: v for k, v in live["report"].items()
                  if k != "season_invariants"}, indent=1))

players.drop(columns=["depth_chart_position"], errors="ignore", inplace=True)
players = players.merge(live["player_depth_rank"][["team", "player_id",
                                                   "depth_chart_position"]],
                        on=["team", "player_id"], how="left")
players["depth_chart_position"] = (players["depth_chart_position"]
                                   .fillna(DF.UNKNOWN_DEPTH_RANK).astype(int))

players.drop(columns=DF.TEAMMATE_FLAG_COLS + DF.OL_FLAG_COLS + DF.DEF_FLAG_COLS,
             errors="ignore", inplace=True)
players = players.merge(live["teammate_flags"], on="team", how="left")
players = players.merge(live["ol_flags"], on="team", how="left")
players = players.merge(
    live["def_flags"].rename(columns={"team": "opponent_team"}),
    on="opponent_team", how="left")

# A column that is CONSTANT across the slate is the 2025 failure mode. Abort on it.
_flat = [c for c in DF.DEPTH_CONTRACT_COLUMNS if players[c].nunique(dropna=False) < 2]
if _flat:
    raise RuntimeError(f"depth/availability columns constant across the slate: {_flat} - "
                       "this is the schema-mismatch failure mode, refusing to project")
print("Depth contract refreshed:",
      {c: int(players[c].nunique()) for c in DF.DEPTH_CONTRACT_COLUMNS})

# -- Recompute team rankings from current rolling values -----------------------
# Use most recent team-level row from hist. Ranks are computed across ALL teams in
# team_metrics (all 32 that appear in hist) BEFORE filtering to active this week -
# this matches the training-time groupby(['season','week']) rank scope.
team_metrics = (
    hist[['team', 'season', 'week', 'off_epa_roll4', 'opp_win_pct_roll4']]
    .drop_duplicates(subset=['team', 'season', 'week'])
    .sort_values(['team', 'season', 'week'])
    .groupby('team').last()
    .reset_index()[['team', 'off_epa_roll4', 'opp_win_pct_roll4']]
    .rename(columns={'off_epa_roll4': '_epa', 'opp_win_pct_roll4': '_sos'})
)
team_metrics['off_epa_rank'] = team_metrics['_epa'].rank(ascending=False, method='min').astype(int)
team_metrics['sos_rank']     = team_metrics['_sos'].rank(ascending=False, method='min').astype(int)
active = team_metrics[team_metrics['team'].isin(players['team'].unique())].copy()

players.drop(columns=['off_epa_roll4', 'opp_win_pct_roll4', 'off_epa_rank', 'sos_rank'],
             errors='ignore', inplace=True)
players = players.merge(active[['team', '_epa', '_sos', 'off_epa_rank', 'sos_rank']],
                        on='team', how='left')
players.rename(columns={'_epa': 'off_epa_roll4', '_sos': 'opp_win_pct_roll4'}, inplace=True)
print(f"Rankings recomputed across {len(active)} teams playing this week.")

# Drop players officially ruled Out
before = len(players)
players = players[players["injury_status_score"].fillna(1.0) > 0.0]
print(f"Dropped {before - len(players)} players ruled Out.")


## Step 6 — Generate Projections

Runs the main position model for each position to produce `projected_pts` (half-PPR total).
For positions with per-stat models, also generates prop columns:

| Position | Prop columns added |
|---|---|
| QB | `pred_qb_pass_yards`, `pred_qb_rush_yards` |
| RB | `pred_rush_yards`, `pred_rec_yards` |
| WR | `pred_wr_receptions`, `pred_wr_rec_yards` |
| TE | `pred_te_receptions`, `pred_te_rec_yards` |

All prop predictions are clipped at 0 (no negative yards/receptions).

In [ ]:
# ── Score with position models ────────────────────────────────────────────────
all_proj = []

for pos in POSITIONS:
    if POS_FILTER and pos != POS_FILTER:
        continue
    pos_players = players[players["position"] == pos].copy()
    if pos_players.empty:
        continue

    feat_cols = models[pos]["feature_cols"]
    missing   = [c for c in feat_cols if c not in pos_players.columns]
    if missing:
        # A feature the model was TRAINED on cannot be invented at serving time.
        # Filling zeros silently changes the feature contract - that is how the
        # 2025 depth block became constant without anyone noticing.
        raise RuntimeError(
            f"{pos}: {len(missing)} required model feature(s) absent from the "
            f"serving frame: {missing} - rebuild features_dataset.csv / fix the "
            "live refresh instead of filling values")

    # Availability cols default to 1.0 (healthy/unknown), not 0 (Out)
    _avail_cols = [c for c in feat_cols if "availability" in c and c in pos_players.columns]
    for _ac in _avail_cols:
        pos_players[_ac] = pos_players[_ac].fillna(1.0)

    # Use training-set medians as fixed fill values (avoids inference-batch drift)
    _hist_pos = hist[(hist["position"] == pos) & (hist["season"] <= 2024)]
    _train_med = _hist_pos[[c for c in feat_cols if c in _hist_pos.columns]].median()
    X = pos_players[feat_cols].fillna(_train_med).fillna(0)
    pos_players = pos_players.copy()
    pos_players["projected_pts"] = models[pos]["model"].predict(X).round(2)

    # QB per-stat breakdown
    if pos == "QB" and QB_STAT_MODELS:
        for stat, stat_model in QB_STAT_MODELS.items():
            sc = stat_model["feature_cols"]
            _ts = _hist_pos[[c for c in sc if c in _hist_pos.columns]].median()
            Xs = pos_players[sc].fillna(_ts).fillna(0)
            pos_players[f"pred_qb_{stat}"] = np.clip(stat_model["model"].predict(Xs), 0, None).round(2)

    # RB per-stat breakdown
    if pos == "RB" and RB_STAT_MODELS:
        for stat, stat_model in RB_STAT_MODELS.items():
            sc = stat_model["feature_cols"]
            _ts = _hist_pos[[c for c in sc if c in _hist_pos.columns]].median()
            Xs = pos_players[sc].fillna(_ts).fillna(0)
            pos_players[f"pred_{stat}"] = np.clip(stat_model["model"].predict(Xs), 0, None).round(2)

    # WR per-stat breakdown
    if pos == "WR" and WR_STAT_MODELS:
        for stat, stat_model in WR_STAT_MODELS.items():
            sc = stat_model["feature_cols"]
            _ts = _hist_pos[[c for c in sc if c in _hist_pos.columns]].median()
            Xs = pos_players[sc].fillna(_ts).fillna(0)
            pos_players[f"pred_wr_{stat}"] = np.clip(stat_model["model"].predict(Xs), 0, None).round(2)

    # TE per-stat breakdown
    if pos == "TE" and TE_STAT_MODELS:
        for stat, stat_model in TE_STAT_MODELS.items():
            sc = stat_model["feature_cols"]
            _ts = _hist_pos[[c for c in sc if c in _hist_pos.columns]].median()
            Xs = pos_players[sc].fillna(_ts).fillna(0)
            pos_players[f"pred_te_{stat}"] = np.clip(stat_model["model"].predict(Xs), 0, None).round(2)

    all_proj.append(pos_players)

if not all_proj:
    raise ValueError("No projections generated, check that features_dataset.csv is up to date.")

proj = pd.concat(all_proj, ignore_index=True)
# Load breakout probabilities (produced by breakout_players.ipynb)
_bko_path = _DIR / 'breakout' / f'breakout_probs_{TARGET_SEASON}.csv'
if _bko_path.exists():
    _bko = pd.read_csv(_bko_path, usecols=['player_id', 'breakout_prob', 'breakout_type'])
    proj = proj.merge(_bko, on='player_id', how='left')
    proj['breakout_prob'] = proj['breakout_prob'].round(3)
    _n_matched = proj['breakout_prob'].notna().sum()
    print(f"Breakout probs loaded: {_n_matched}/{len(proj)} players matched "
          f"(from {_bko_path.name})")
else:
    proj['breakout_prob'] = float('nan')
    proj['breakout_type'] = ''
    print(f"Note: {_bko_path.name} not found — run breakout_players.ipynb first.")



# ── Print + save ──────────────────────────────────────────────────────────────
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 120)

SEP = "=" * 65
print(SEP)
print(f"  Season {TARGET_SEASON}  Week {TARGET_WEEK}  Half-PPR Projections")
print(SEP)

for pos in POSITIONS:
    if POS_FILTER and pos != POS_FILTER:
        continue
    subset = (
        proj[proj["position"] == pos]
        .sort_values("projected_pts", ascending=False)
        .head(20)
        [["player_display_name", "team", "opponent_team", "projected_pts",
          "implied_team_total", "depth_chart_position", "injury_status_score"]]
        .reset_index(drop=True)
    )
    if subset.empty:
        continue
    subset.index += 1
    print()
    print(f"--- {pos} (Top 20) ---")
    print(subset.to_string())

qb_stat_cols = [f"pred_qb_{s}" for s in QB_STAT_NAMES if f"pred_qb_{s}" in proj.columns]
rb_stat_cols = [f"pred_{s}" for s in RB_STAT_NAMES if f"pred_{s}" in proj.columns]
wr_stat_cols = [f"pred_wr_{s}" for s in WR_STAT_NAMES if f"pred_wr_{s}" in proj.columns]
te_stat_cols = [f"pred_te_{s}" for s in TE_STAT_NAMES if f"pred_te_{s}" in proj.columns]
out_path = _DIR / "fantasy_projections" / f"projections_{TARGET_SEASON}_week{TARGET_WEEK:02d}.csv"
os.makedirs(out_path.parent, exist_ok=True)
_base_cols = ["player_id", "player_display_name", "position", "team", "opponent_team",
              "gameday", "projected_pts", "implied_team_total",
              "depth_chart_position", "injury_status_score", "is_home",
              "off_epa_roll4", "opp_season_win_pct", "opp_win_pct_roll4",
              "off_epa_rank", "sos_rank"]
# app.py depends on these base cols, so fail loud here rather than silently dropping one
# (or emitting a cryptic KeyError) if features_dataset.csv was rebuilt without it.
_missing_out = [c for c in _base_cols if c not in proj.columns]
assert not _missing_out, (f"projection output missing expected column(s) {_missing_out} -- "
                          "rebuild features_dataset.csv (features.ipynb) so they are present.")
_out_cols = (_base_cols + qb_stat_cols + rb_stat_cols + wr_stat_cols + te_stat_cols
             + (["breakout_prob", "breakout_type"] if "breakout_prob" in proj.columns else []))
proj[_out_cols].sort_values(
    ["position", "projected_pts"], ascending=[True, False]
).to_csv(out_path, index=False)
print()
print(f"Saved: {out_path}")


## Step 7 — Projection Analysis

Summarizes this week’s projections:
- **Distribution** of projected half-PPR points by position (min / 25th / median / 75th / max)
- **Prop stat leaders** by category (rush yards, rec yards, receptions, pass yards)
- **Position scorecards** showing top 10 per position with inline prop stats

Run this cell after Step 6 to inspect the projections before the week is played.

In [25]:
SEP2 = "-" * 65

# ── Distribution summary ─────────────────────────────────────────
print("PROJECTED PTS DISTRIBUTION")
print(SEP2)
dist = (
    proj.groupby("position")["projected_pts"]
    .describe(percentiles=[0.25, 0.5, 0.75])
    [["count", "min", "25%", "50%", "75%", "max", "mean"]]
    .round(1)
)
print(dist.to_string())
print()

# ── Prop stat leaders ────────────────────────────────────────────
print("PROP STAT LEADERS")
print(SEP2)

def top_n(df, col, label, n=5):
    if col not in df.columns:
        return
    rows = df[df[col].notna()].nlargest(n, col)[["player_display_name", "team", col]]
    if rows.empty:
        return
    print(f"{label}:")
    for _, r in rows.iterrows():
        print(f"  {r['player_display_name']:<22} {r['team']:<4} {r[col]:>6.1f}")
    print()

top_n(proj, "pred_qb_pass_yards",  "QB Pass Yards")
top_n(proj, "pred_qb_rush_yards",  "QB Rush Yards")
top_n(proj, "pred_rush_yards",     "RB Rush Yards")
top_n(proj, "pred_rec_yards",      "RB Rec Yards")
top_n(proj, "pred_wr_receptions",  "WR Receptions")
top_n(proj, "pred_wr_rec_yards",   "WR Rec Yards")
top_n(proj, "pred_te_receptions",  "TE Receptions")
top_n(proj, "pred_te_rec_yards",   "TE Rec Yards")

# ── Top 10 per position with inline prop stats ───────────────────
print("TOP 10 PER POSITION (with prop stats)")
print(SEP2)

pos_prop_cols = {
    "QB": [("pred_qb_pass_yards", "PassYds"), ("pred_qb_rush_yards", "RushYds")],
    "RB": [("pred_rush_yards",    "RushYds"), ("pred_rec_yards",    "RecYds")],
    "WR": [("pred_wr_receptions", "Rec"),     ("pred_wr_rec_yards", "RecYds")],
    "TE": [("pred_te_receptions", "Rec"),     ("pred_te_rec_yards", "RecYds")],
}

for pos in ["QB", "RB", "WR", "TE"]:
    if POS_FILTER and pos != POS_FILTER:
        continue
    subset = proj[proj["position"] == pos].nlargest(10, "projected_pts").reset_index(drop=True)
    if subset.empty:
        continue
    prop_defs = pos_prop_cols[pos]
    print(f"{pos}")
    header = f"  {'#':>2}  {'Player':<22} {'Team':<4} {'Opp':<4} {'ProjPts':>7}"
    for prop_col, label in prop_defs:
        if prop_col in subset.columns:
            header += f"  {label:>7}"
    print(header)
    for i, row in subset.iterrows():
        line = f"  {i+1:>2}  {str(row.get('player_display_name','')):<22} {str(row.get('team','')):<4} {str(row.get('opponent_team','')):<4} {row['projected_pts']:>7.1f}"
        for col, _ in prop_defs:
            if col in subset.columns and not pd.isna(row.get(col)):
                line += f"  {row[col]:>7.1f}"
        print(line)
    print()


PROJECTED PTS DISTRIBUTION
-----------------------------------------------------------------
          count  min  25%   50%   75%   max  mean
position                                         
QB         76.0  1.2  7.0  13.4  16.3  24.1  12.2
RB        143.0  0.4  2.7   4.6   8.1  19.1   5.8
TE        123.0  0.4  1.7   2.6   5.2  12.7   3.7
WR        226.0  0.2  2.2   3.8   6.1  19.2   4.6

PROP STAT LEADERS
-----------------------------------------------------------------
QB Pass Yards:
  Brock Purdy            SF    282.7
  Matthew Stafford       LA    277.2
  Philip Rivers          IND   271.9
  Jared Goff             DET   269.8
  Jordan Love            GB    262.0

QB Rush Yards:
  Anthony Richardson     IND    47.7
  Justin Fields          NYJ    36.4
  Josh Allen             BUF    33.3
  Drake Maye             NE     32.1
  Jalen Hurts            PHI    29.5

RB Rush Yards:
  James Cook             BUF    88.9
  De'Von Achane          MIA    88.1
  Derrick Henry          BAL   

## Model Performance Summary — 2025 Season (Weeks 10–17)

### Fantasy Points

| Position | MAE | Bias | Correlation | Top-12 Hit Rate |
|----------|-----|------|-------------|------------------|
| QB | 6.28 pts | +0.72 (slight over) | 0.51 | 54% |
| RB | 4.07 pts | -0.06 (near zero) | 0.68 | 50% |
| WR | 3.59 pts | +0.31 (near zero) | 0.62 | 30% |
| TE | 2.88 pts | +0.33 (near zero) | 0.63 | 48% |

**RB and TE are the strongest.** Near-zero bias and solid correlation — the model knows who the good players are and doesn't systematically over/under-project them. **WR is the weakest** — MAE looks fine but the 30% top-12 hit rate reflects how volatile WR target share is week to week. **QB is respectable** — slightly over-projects due to blowouts where starters get pulled, but 54% top-12 hit rate is decent.

### Prop Stat Models

| Stat | MAE | Bias | Usability |
|------|-----|------|-----------|
| RB Rec Yards | 9.9 yds | +0.5 | ✅ Best prop model — lines typically set at 15–25 yds |
| TE Rec Yards | 14.4 yds | +2.1 | ✅ Solid — lines usually 25–50 yds |
| RB Rush Yards | 18.6 yds | -0.5 | ✅ Usable — lines 40–80 yds depending on back |
| WR Receptions | 1.2 rec | +0.1 | ✅ Directionally useful for half-point lines |
| WR Rec Yards | 19.3 yds | +1.4 | ⚠️ High variance — use directionally only |
| TE Receptions | 1.2 rec | +0.1 | ✅ Solid for half-point lines |
| QB Rush Yards | 11.1 yds | +0.1 | ✅ Near-zero bias |
| QB Pass Yards | 70.8 yds | +25.1 | ❌ Systematic over-projection — avoid betting |

**Best prop bets to reference:** RB rec yards, TE rec yards, RB rush yards. These have low MAE and near-zero bias relative to typical book lines. **Avoid QB pass yards** — the +25 yard over-projection bias is too large and too consistent to use confidently.
